> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 10 · Notebook 04 — Supervised models, calibration and regimes

**Sessions:** S7–S8 (Supervised models · Unsupervised learning & market regimes) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Compare a logistic regression, a random forest and gradient boosting with the best possible model.
2. See how class weights break probabilities, and fix them with calibration.
3. Find hidden regimes with a Gaussian mixture and a hidden Markov model.
4. Tell a smoothed regime label from a real-time one.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()

## 1. Three models against the truth

`p.planted_features()` has one real `signal` with a known probability `P(y = 1) = 1/(1 + e^{−1.5·signal})`, a `twin` (signal + noise) and two noise features. Knowing the true probability gives the best possible AUC, so we can see how close each model gets.

In [ ]:
from sklearn.metrics import roc_auc_score

X, y = p.planted_features()
true_p = 1 / (1 + np.exp(-1.5 * X["signal"]))
tr, te = slice(0, 1400), slice(1400, 2000)
rows = {"best possible (true probability)": {"AUC": roc_auc_score(y.iloc[te], true_p.iloc[te]), "mean |p − true p|": 0.0}}
for kind in ("logit", "rf", "gbm"):
    prob = p.make_model(kind).fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
    rows[kind] = {"AUC": roc_auc_score(y.iloc[te], prob), "mean |p − true p|": np.abs(prob - true_p.iloc[te]).mean()}
pd.DataFrame(rows).T.round(3)

The logistic regression is almost perfect: the truth *is* logistic. The forest and the boosted trees are close in AUC but their probabilities are further from the truth. More flexible is not better when the signal is simple.

## 2. Probabilities you can size with

A crash-type label is rare (about 10% here). A common trick for imbalanced labels, `class_weight="balanced"`, makes the model predict far too often: its "probabilities" average about 0.35. Bet sizing on them would be badly wrong. **Calibrate** on a separate, later sample (`p.calibrate`, isotonic). Then check with a **reliability table**: equal-width probability bins on [0, 1] (`bins = 5` → bin = `min(int(p·5), 4)`), and per non-empty bin the mean prediction, the observed frequency and the count.

In [ ]:
from sklearn.linear_model import LogisticRegression

Xr, _ = p.planted_features(4000, seed=3)
true_rare = 1 / (1 + np.exp(-(1.5 * Xr["signal"] - 3)))
y_rare = pd.Series((np.random.default_rng(5).random(len(Xr)) < true_rare).astype(int), index=Xr.index)
model = LogisticRegression(class_weight="balanced").fit(Xr.iloc[:2000], y_rare.iloc[:2000])
calibrated = p.calibrate(model, Xr.iloc[2000:3000], y_rare.iloc[2000:3000])
y_test = y_rare.iloc[3000:]
prob_raw = model.predict_proba(Xr.iloc[3000:])[:, 1]
prob_cal = calibrated.predict_proba(Xr.iloc[3000:])[:, 1]

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def reliability_table(prob, y, bins=5):
    prob, y = np.asarray(prob, dtype=float), np.asarray(y, dtype=float)
    b = np.minimum((prob * bins).astype(int), bins - 1)
    g = pd.DataFrame({"b": b, "p": prob, "y": y}).groupby("b")
    return pd.DataFrame({"mean_pred": g["p"].mean(), "frac_pos": g["y"].mean(), "count": g.size()})

cases = [(prob_raw, y_test), (prob_cal, y_test), ([0.05, 0.15, 0.35, 0.95, 1.0], [0, 1, 0, 1, 1])]
mine = [p.attempt(reliability_table, *c) for c in cases]
mine = p.check("reliability_table", mine, [p.reliability_table(*c) for c in cases])
print(f"base rate {y_test.mean():.3f}; mean predicted probability: raw {prob_raw.mean():.3f}, calibrated {prob_cal.mean():.3f};  "
      f"AUC raw {roc_auc_score(y_test, prob_raw):.3f}, calibrated {roc_auc_score(y_test, prob_cal):.3f}")
display(pd.concat({"raw (balanced weights)": mine[0], "calibrated (isotonic)": mine[1]}, axis=1).round(3))

Raw, a predicted 0.9 happens less than half the time. Calibrated, the mean prediction matches the base rate and each well-populated bin's prediction is close to what happens (the top bins hold only a few dozen predictions, so they are noisy). Calibration barely changes the ranking (the AUC); it makes the numbers usable for sizing.

## 3. Regimes

The regime market hides a calm trending state and a volatile choppy one. A **Gaussian mixture** on rolling volatility and efficiency clusters days without any notion of time. A **hidden Markov model** adds persistence: today's state depends on yesterday's. `p.hmm_fit` fits a 2-state Gaussian HMM on the returns (Baum–Welch). Its **forward filter** gives `P(state_t | returns up to t)`, a real-time signal: from yesterday's probabilities, predict with the transition matrix and weight by today's likelihood, `a = (out[t−1] @ A) × B[t]`, then normalize.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def hmm_filter(x, params):
    x = np.asarray(x, dtype=float)
    B = p._normal_pdf(x, params["mu"], params["var"]) + 1e-300      # likelihood of each observation in each state
    out = np.empty_like(B)
    a = params["pi"] * B[0]
    out[0] = a / a.sum()
    for t in range(1, len(x)):
        a = (out[t - 1] @ params["A"]) * B[t]
        out[t] = a / a.sum()
    return out

mk = p.regime_market()
r = np.log(mk["close"]).diff().dropna()
params = p.hmm_fit(r.to_numpy())
mine = p.check("hmm_filter", p.attempt(hmm_filter, r.to_numpy(), params), p.hmm_filter(r.to_numpy(), params))
print("state volatilities:", np.round(np.sqrt(params["var"]), 4), " stay probabilities:", np.round(np.diag(params["A"]), 3))

In [ ]:
rr = np.log(mk["close"]).diff()
feats = pd.DataFrame({"vol20": rr.rolling(20).std(), "er20": rr.rolling(20).sum().abs() / rr.abs().rolling(20).sum()})
calm_truth = (mk["regime"] == 1).astype(int)                      # the hidden trend regime is the calm one
calm_state = int(np.argmin(params["var"]))
labels = {"GMM (vol20, er20)": (p.gmm_regimes(feats) == 0).astype(int),
          "HMM, Viterbi (whole sample)": (p.hmm_regimes(rr) == 0).astype(int),
          "HMM, forward filter (real time)": pd.Series((mine[:, calm_state] > 0.5).astype(int), index=r.index)}
pd.DataFrame({k: {"agreement with the hidden regime": (v == calm_truth.loc[v.index]).mean(),
                  "regime switches": int((v.diff() != 0).sum())} for k, v in labels.items()}
             | {"truth": {"agreement with the hidden regime": 1.0, "regime switches": int((mk["regime"].diff() != 0).sum())}}).T.round(3)

The GMM gets most days right but flickers; the HMM's Viterbi path is both more accurate and about as persistent as the truth. But Viterbi uses the **whole** sample, future included: it describes the past, it cannot be traded. The real-time filter is still good, with more switches. Does the regime matter for a strategy?

In [ ]:
momentum = p.momentum_side(mk["close"]).shift(1) * rr                 # yesterday's signal, today's return
p.regime_performance(momentum, labels["HMM, forward filter (real time)"].map({1: "calm", 0: "volatile"})).round(3)

Momentum earns in the calm state and loses in the volatile one, as the hidden regimes were built to do. Here a real-time regime label is a useful **feature** (or a filter for the meta-model).

## Wrap-up

* Start with the simplest model that fits the signal; flexibility costs variance.
* Class weights and many models give scores, not probabilities: calibrate on later data before sizing.
* Regime labels from smoothers (Viterbi) look great and use the future; use filtered probabilities in trading.
* Graded version: `labs/part10/week34_labels` (`make_model`, `calibrate`, `reliability_table`, `gmm_regimes`, `hmm_regimes` with hmmlearn).